# uv (Astral) — What it is, why it matters, and exactly how to use it

`uv` is a **fast, batteries‑included Python package & project manager** from Astral (makers of Ruff). It unifies tasks you’d normally split across tools like `pip`, `pip-tools`, `pipx`, `virtualenv`, `poetry`, and `pyenv`:

* **Project management**: `pyproject.toml` + a **cross‑platform lockfile** (`uv.lock`) for reproducible installs.
* **Pip‑compatible interface**: `uv pip …` as a drop‑in speed boost for existing `pip`/`requirements.txt` workflows.
* **Virtualenvs & Python versions**: `uv venv` and `uv python` to create envs and install/switch Python versions.
* **Script & tool runner**: `uv run` to execute commands in a consistent env; `uvx` / `uv tool …` for CLI tools (like `ruff`, `black`) without polluting your project.
* **Global cache + offline‑friendly**: aggressively caches wheels & metadata, supports `--offline`, and precise cache control.

> **Mental model**: In “project mode”, `uv` keeps three things in sync: your `pyproject.toml` (declared deps), `uv.lock` (resolved pins for all platforms), and `.venv` (installed packages). In “pip‑compat mode”, it behaves like a faster `pip`/`pip-tools`.

---

## Install uv (once)

On macOS or Linux:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

On Windows (PowerShell):

```powershell
powershell -ExecutionPolicy Bypass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Check:

```bash
uv --version
```

---

## The two big ways to use uv

### 1) **Project workflow** (recommended for new projects)

* You declare deps in **`pyproject.toml`**.
* uv resolves to **`uv.lock`** (universal lock, cross‑platform by default).
* uv installs into **`.venv/`** and runs commands in that **consistent, locked** env.

### 2) **Pip‑compatible workflow** (keep `requirements*.txt`)

* Use `uv pip` as a fast `pip` replacement.
* Optionally use `uv pip compile` / `uv pip sync` as a faster `pip-tools` alternative.

> You can mix both in one repo when needed, but stick to **one primary workflow** per project to avoid confusion.

---

## Quickstart A — New project (project workflow)

```bash
# 1) Create a new project scaffold
uv init myapp
cd myapp

# 2) (Optional) Pin the Python version for this project
uv python pin 3.12

# 3) Add runtime deps
uv add fastapi uvicorn

# 4) Add dev tools into a separate group
uv add --group dev ruff pytest

# 5) Run a command inside the project env (auto-syncs lock & .venv)
uv run uvicorn main:app --reload
```

What you now have:

* `pyproject.toml` — your declared deps (and groups like `dev`).
* `uv.lock` — exact, **cross‑platform** resolution of every transitive dep.
* `.venv/` — installed packages for your current machine.

Useful follow‑ups:

```bash
# Install everything according to the lockfile (including dev deps by default)
uv sync

# Production-style install (exclude dev)
uv sync --no-group dev

# Only include specific groups
uv sync --only-group docs   # installs only the docs group (excludes project itself)

# Remove a dependency
uv remove uvicorn

# Build & publish
uv build
uv publish  # needs credentials configured
```

**Why this is nice**

* `uv run` **always ensures** the env matches the lock (no drift).
* The lockfile is universal: teams on Linux/macOS/Windows get the **same** versions resolved.

---

## Quickstart B — Keep `requirements.txt` (pip‑compatible)

If you can’t switch to `pyproject.toml` yet, use `uv` as a faster `pip`/`pip‑tools`:

```bash
# Create a venv (or use your existing one)
uv venv
source .venv/bin/activate   # PowerShell: .venv\Scripts\Activate.ps1

# Install from a requirements file
uv pip install -r requirements.txt

# Compile locked requirements (like pip-compile), output to requirements.txt
uv pip compile pyproject.toml -o requirements.txt   # or: requirements.in -> requirements.txt

# Sync your env exactly to a lockfile (like pip-sync)
uv pip sync requirements.txt
```

> By default `uv pip` wants a **virtualenv**. If you truly need to modify the **system** Python (e.g. in CI/containers), use `--system` or set `UV_SYSTEM_PYTHON=true` environment‑wide. Prefer virtualenvs for local dev.

---

## Quickstart C — One‑off scripts (inline deps)

`uv` can manage dependencies per‑file:

```bash
# Create a script
printf 'import requests;print(requests.get("https://astral.sh").status_code)\n' > fetch.py

# Attach a dependency to the script (stored as inline metadata)
uv add --script fetch.py requests

# Run in an isolated, auto-managed env
uv run fetch.py
```

You can also bring in a package ad‑hoc without editing project files:

```bash
uv run --with "pandas>=2.2" python -c "import pandas as pd; print(pd.__version__)"
```

---

## Quickstart D — Tools with `uvx` and `uv tool`

Run a CLI tool at a given version **without** installing it globally:

```bash
uvx ruff@latest check .
uvx black@24.10.0 --help
```

Make tools persistent on your PATH (like `pipx`):

```bash
uv tool install ruff
ruff --version

# Upgrade or uninstall later
uv tool upgrade ruff
uv tool uninstall ruff
```

> Tip: `uvx` uses ephemeral, cached envs. `uv tool install` creates a reusable install in your user space.

---

## Using uv with Jupyter/Notebooks

Install Jupyter in your project and register a kernel named after the project:

```bash
# In your project
uv add --group dev jupyter ipykernel
uv run python -m ipykernel install --user --name myapp --display-name "Python (myapp)"

# Launch Jupyter with the project’s env
uv run jupyter lab   # or: uv run jupyter notebook
```

> In VS Code, select the **Python (myapp)** kernel to run notebooks inside the project’s `.venv`.

---

## Everyday commands you’ll actually use

**Dependencies**

```bash
uv add requests            # add to default (runtime) deps
uv add --group dev pytest  # dev-only
uv remove requests
```

**Sync & lock**

```bash
uv sync              # bring .venv in line with uv.lock (re-locks first if needed)
uv lock              # update uv.lock from pyproject.toml
uv sync --locked     # install strictly from existing uv.lock (fail if re-lock needed)
uv sync --frozen     # don’t modify uv.lock; use it as-is
```

**Groups & extras**

```bash
uv sync --no-group dev           # typical production install
uv sync --group docs             # include only docs deps (still includes project)
uv run --extra gpu python app.py # run with optional extra
```

**Python & virtualenvs**

```bash
uv python install 3.13     # install a Python version (downloaded by uv)
uv python list             # show installed & available Pythons
uv python pin 3.12         # write .python-version for this project
uv venv --python 3.12      # create .venv usi
```
